# Credit Card Fraud Detection Using Artificial Neural Networks (Deep Learning)
### End-to-End Academic Research & Engineering Notebook

**Objective:** Build an Artificial Neural Network (ANN) using TensorFlow/Keras to accurately detect fraudulent credit card transactions within an extremely skewed class distribution (~0.172% fraud), applying SMOTE resampling strictly without data leakage.

---

## 1. Setup & Imports

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    roc_curve,
    precision_recall_curve
)
from imblearn.over_sampling import SMOTE

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

print(f"TensorFlow Version: {tf.__version__}")

## 2. Dataset Ingestion & Exploratory Inspection

In [ ]:
DATA_PATH = os.path.join("..", "data", "creditcard.csv")
if not os.path.exists(DATA_PATH):
    print("Dataset not found at data/creditcard.csv. Please download from Kaggle.")
else:
    df = pd.read_csv(DATA_PATH)
    print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")
    print("\nMissing values:", df.isnull().sum().max())
    print("Duplicate rows:", df.duplicated().sum())
    print("\nClass balance:")
    print(df['Class'].value_counts())
    print(f"Fraud Percentage: {df['Class'].mean() * 100:.4f}%")
    df.head()

## 3. Data Preprocessing & Leakage-Free Scaling
- Separate features ($X$) and target ($y$)
- Stratified Train/Test Split (80/20)
- Fit `RobustScaler` on training set only
- Apply SMOTE on training split only

In [ ]:
X = df.drop(columns=['Class'])
y = df['Class']

# Stratified train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale Amount and Time strictly on training data
scaler_amt = RobustScaler()
scaler_time = RobustScaler()

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled['Amount'] = scaler_amt.fit_transform(X_train[['Amount']])
X_train_scaled['Time'] = scaler_time.fit_transform(X_train[['Time']])

X_test_scaled['Amount'] = scaler_amt.transform(X_test[['Amount']])
X_test_scaled['Time'] = scaler_time.transform(X_test[['Time']])

# SMOTE on training split only
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train_scaled, y_train)

print(f"Training set before SMOTE: {len(X_train)} (Frauds: {y_train.sum()})")
print(f"Training set after SMOTE: {len(X_train_res)} (Frauds: {y_train_res.sum()})")
print(f"Test set (Untouched): {len(X_test)} (Frauds: {y_test.sum()})")

## 4. Artificial Neural Network (ANN) Architecture
Architecture specified:
- Input Layer: 30 features
- Dense Layer: 128 neurons, ReLU + BatchNormalization + Dropout(0.3)
- Dense Layer: 64 neurons, ReLU + BatchNormalization + Dropout(0.2)
- Dense Layer: 32 neurons, ReLU
- Output Layer: 1 neuron, Sigmoid
- Optimizer: Adam, Loss: Binary Crossentropy

In [ ]:
model = models.Sequential([
    layers.Input(shape=(30,)),
    layers.Dense(128, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(64, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.2),
    layers.Dense(32, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.Precision(name='precision'), tf.keras.metrics.Recall(name='recall'), tf.keras.metrics.AUC(name='auc')]
)
model.summary()

## 5. Model Training with EarlyStopping & ModelCheckpoint

In [ ]:
early_stop = callbacks.EarlyStopping(
    monitor='val_loss',
    patience=6,
    restore_best_weights=True
)

history = model.fit(
    X_train_res,
    y_train_res,
    validation_data=(X_test_scaled.values, y_test.values),
    epochs=20,
    batch_size=1024,
    callbacks=[early_stop],
    verbose=1
)

## 6. Evaluation on Untouched Test Set

In [ ]:
y_pred_probs = model.predict(X_test_scaled.values, batch_size=2048).flatten()
y_pred = (y_pred_probs >= 0.5).astype(int)

print(classification_report(y_test, y_pred, digits=4))
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_pred_probs):.4f}")
print(f"PR-AUC Score:  {average_precision_score(y_test, y_pred_probs):.4f}")

# Confusion Matrix Plot
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Genuine', 'Fraud'], yticklabels=['Genuine', 'Fraud'])
plt.title('Test Set Confusion Matrix')
plt.ylabel('Actual Class')
plt.xlabel('Predicted Class')
plt.show()